# Automated Brain Tumor Segmentation and Classification using U-Net and ResNet

This notebook demonstrates an end-to-end deep learning framework for brain tumor segmentation and classification.

## 1. Setup and Imports

In [ ]:
import os
import numpy as np
import cv2
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models, applications

## 2. Dataset Loading and Preprocessing
Note: If masks are missing in the dataset, pseudo-masks are generated using thresholding for demonstration purposes.

In [ ]:
# Placeholder for data loading
def load_data(img_dir, mask_dir=None):
    print(f"Loading data from {img_dir}...")
    # Generate pseudo masks if mask_dir is None
    return np.random.normal(0, 1, (10, 256, 256, 3)), np.random.randint(0, 2, (10, 256, 256, 1))
X_train, y_train_mask = load_data('../Dataset/Training/')

## 3. Model 1: Custom CNN Backbone + Vanilla U-Net (Segmentation)

In [ ]:
def build_vanilla_unet(input_shape=(256, 256, 3)):
    inputs = layers.Input(shape=input_shape)
    # Encoder
    c1 = layers.Conv2D(32, (3, 3), activation='relu', padding='same')(inputs)
    p1 = layers.MaxPooling2D((2, 2))(c1)
    # Bottleneck
    c2 = layers.Conv2D(64, (3, 3), activation='relu', padding='same')(p1)
    # Decoder
    u1 = layers.Conv2DTranspose(32, (2, 2), strides=(2, 2), padding='same')(c2)
    c3 = layers.Conv2D(32, (3, 3), activation='relu', padding='same')(u1)
    outputs = layers.Conv2D(1, (1, 1), activation='sigmoid')(c3)
    return models.Model(inputs, outputs)

model1 = build_vanilla_unet()
model1.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model1.summary()

## 4. Model 2: ResNet50 + U-Net Transfer Learning (Segmentation)

In [ ]:
def build_resnet50_unet(input_shape=(256, 256, 3)):
    resnet = applications.ResNet50(weights='imagenet', include_top=False, input_shape=input_shape)
    inputs = resnet.input
    # Using a shallow decoder for demonstration
    x = layers.Conv2DTranspose(128, (2, 2), strides=(2, 2), padding='same')(resnet.output)
    x = layers.Conv2DTranspose(64, (2, 2), strides=(2, 2), padding='same')(x)
    x = layers.Conv2DTranspose(32, (2, 2), strides=(2, 2), padding='same')(x)
    x = layers.Conv2DTranspose(16, (2, 2), strides=(2, 2), padding='same')(x)
    x = layers.Conv2DTranspose(16, (2, 2), strides=(2, 2), padding='same')(x)
    outputs = layers.Conv2D(1, (1, 1), activation='sigmoid')(x)
    return models.Model(inputs, outputs)

model2 = build_resnet50_unet()
model2.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

## 5. Model 3: VGG16 Custom Classifier (Classification)

In [ ]:
def build_vgg16_classifier(input_shape=(256, 256, 3), num_classes=4):
    vgg = applications.VGG16(weights='imagenet', include_top=False, input_shape=input_shape)
    x = layers.Flatten()(vgg.output)
    x = layers.Dense(128, activation='relu')(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)
    return models.Model(vgg.input, outputs)

model3 = build_vgg16_classifier()
model3.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

## 6. Model 4: ResNet101 Unified Multi-Task Network (Joint Segmentation & Classification)

In [ ]:
def build_resnet101_multitask(input_shape=(256, 256, 3), num_classes=4):
    resnet = applications.ResNet101(weights='imagenet', include_top=False, input_shape=input_shape)
    inputs = resnet.input
    
    # Classification Branch
    c_branch = layers.GlobalAveragePooling2D()(resnet.output)
    class_out = layers.Dense(num_classes, activation='softmax', name='class_out')(c_branch)
    
    # Segmentation Branch (simplified)
    s_branch = layers.Conv2DTranspose(128, (2, 2), strides=(2, 2), padding='same')(resnet.output)
    for _ in range(4): # upsample back to 256x256
        s_branch = layers.Conv2DTranspose(64, (2, 2), strides=(2, 2), padding='same')(s_branch)
    seg_out = layers.Conv2D(1, (1, 1), activation='sigmoid', name='seg_out')(s_branch)
    
    return models.Model(inputs, [class_out, seg_out])

model4 = build_resnet101_multitask()
model4.compile(optimizer='adam', 
              loss={'class_out': 'sparse_categorical_crossentropy', 'seg_out': 'binary_crossentropy'},
              metrics=['accuracy'])

## 7. Training Loop and Evaluation

In [ ]:
# print("Training demonstration...")
# This code is meant to be run in an environment with GPUs.
# We skip actual training in this notebook as it requires heavy compute.